In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.sparse import csr_matrix
from tqdm import tqdm


# -----------------------------
# 0. Setup
# -----------------------------

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

SEED = 256

N_EVAL_USERS = 1000
EPOCHS = 10
BATCH_SIZE = 8192

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


# -----------------------------
# 1. Load parquet splits
# -----------------------------

train = pd.read_parquet("./bgg_recsys/train.parquet")
val = pd.read_parquet("./bgg_recsys/val.parquet")
test = pd.read_parquet("./bgg_recsys/test.parquet")

print(train.head())
print(train.columns)

cols = ["user", "game_id", "name", "rating"]

train = train[cols].dropna().copy()
val = val[cols].dropna().copy()
test = test[cols].dropna().copy()

train = train[train["rating"] > 0].copy()
val = val[val["rating"] > 0].copy()
test = test[test["rating"] > 0].copy()


# -----------------------------
# 2. Encode users/items using train only
# -----------------------------

user_to_idx = {u: i for i, u in enumerate(train["user"].unique())}
game_to_idx = {g: i for i, g in enumerate(train["game_id"].unique())}

idx_to_game = {i: g for g, i in game_to_idx.items()}

train["user_idx"] = train["user"].map(user_to_idx)
train["game_idx"] = train["game_id"].map(game_to_idx)

val["user_idx"] = val["user"].map(user_to_idx)
val["game_idx"] = val["game_id"].map(game_to_idx)

test["user_idx"] = test["user"].map(user_to_idx)
test["game_idx"] = test["game_id"].map(game_to_idx)

val = val.dropna(subset=["user_idx", "game_idx"]).copy()
test = test.dropna(subset=["user_idx", "game_idx"]).copy()

for df in [train, val, test]:
    df["user_idx"] = df["user_idx"].astype(int)
    df["game_idx"] = df["game_idx"].astype(int)

N_USERS = len(user_to_idx)
N_GAMES = len(game_to_idx)

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)
print("Users:", N_USERS)
print("Games:", N_GAMES)


# -----------------------------
# 3. Build user-item matrix
# -----------------------------

user_item_matrix = csr_matrix(
    (
        np.ones(len(train), dtype=np.float32),
        (train["user_idx"], train["game_idx"])
    ),
    shape=(N_USERS, N_GAMES)
)


# -----------------------------
# 4. LightGCN model
# -----------------------------

class LightGCN(nn.Module):
    def __init__(self, n_users, n_items, edge_index, embedding_dim=64, n_layers=3):
        super().__init__()

        self.n_users = n_users
        self.n_items = n_items
        self.n_nodes = n_users + n_items
        self.n_layers = n_layers

        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)

        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

        self.register_buffer("edge_index", edge_index)
        self.norm_adj = self.build_norm_adj(edge_index)

    def build_norm_adj(self, edge_index):
        row, col = edge_index

        deg = torch.bincount(row, minlength=self.n_nodes).float()
        deg_inv_sqrt = torch.pow(deg + 1e-8, -0.5)

        values = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        adj = torch.sparse_coo_tensor(
            edge_index,
            values,
            size=(self.n_nodes, self.n_nodes),
            device=edge_index.device
        )

        return adj.coalesce()

    def propagate(self):
        all_emb = torch.cat(
            [self.user_embedding.weight, self.item_embedding.weight],
            dim=0
        )

        embeddings = [all_emb]

        for _ in range(self.n_layers):
            all_emb = torch.sparse.mm(self.norm_adj, all_emb)
            embeddings.append(all_emb)

        final_emb = torch.stack(embeddings, dim=0).mean(dim=0)

        users_emb = final_emb[:self.n_users]
        items_emb = final_emb[self.n_users:]

        return users_emb, items_emb

    def forward(self, users, pos_items, neg_items):
        users_emb, items_emb = self.propagate()

        u = users_emb[users]
        pos = items_emb[pos_items]
        neg = items_emb[neg_items]

        pos_scores = (u * pos).sum(dim=1)
        neg_scores = (u * neg).sum(dim=1)

        return pos_scores, neg_scores


def build_lightgcn_edges(train_df, n_users):
    users = torch.tensor(train_df["user_idx"].values, dtype=torch.long)
    items = torch.tensor(train_df["game_idx"].values, dtype=torch.long) + n_users

    edge_u_to_i = torch.stack([users, items], dim=0)
    edge_i_to_u = torch.stack([items, users], dim=0)

    edge_index = torch.cat([edge_u_to_i, edge_i_to_u], dim=1)

    return edge_index


edge_index = build_lightgcn_edges(train, N_USERS).to(DEVICE)

model = LightGCN(
    n_users=N_USERS,
    n_items=N_GAMES,
    edge_index=edge_index,
    embedding_dim=64,
    n_layers=3
).to(DEVICE)


# -----------------------------
# 5. Helper data structures
# -----------------------------

global_mean = train["rating"].mean()
rating_min = train["rating"].min()
rating_max = train["rating"].max()

user_pos_items = (
    train.groupby("user_idx")["game_idx"]
    .apply(lambda x: x.values.astype(np.int64))
    .to_dict()
)

user_pos_sets = {
    u: set(items)
    for u, items in user_pos_items.items()
}

all_users = np.array(list(user_pos_items.keys()), dtype=np.int64)


# -----------------------------
# 6. Train LightGCN with BPR loss
# -----------------------------

def sample_bpr_batch(batch_size=4096):
    users = np.random.choice(all_users, size=batch_size, replace=True)

    pos_items = np.array([
        np.random.choice(user_pos_items[u])
        for u in users
    ], dtype=np.int64)

    neg_items = np.random.randint(0, N_GAMES, size=batch_size)

    for i, u in enumerate(users):
        while neg_items[i] in user_pos_sets[u]:
            neg_items[i] = np.random.randint(0, N_GAMES)

    return (
        torch.tensor(users, dtype=torch.long, device=DEVICE),
        torch.tensor(pos_items, dtype=torch.long, device=DEVICE),
        torch.tensor(neg_items, dtype=torch.long, device=DEVICE)
    )


def bpr_loss(pos_scores, neg_scores):
    return -torch.mean(F.logsigmoid(pos_scores - neg_scores))


STEPS_PER_EPOCH = max(1, len(train) // BATCH_SIZE)
LR = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0

    for _ in tqdm(range(STEPS_PER_EPOCH), desc=f"Epoch {epoch}/{EPOCHS}"):
        users, pos_items, neg_items = sample_bpr_batch(BATCH_SIZE)

        pos_scores, neg_scores = model(users, pos_items, neg_items)
        loss = bpr_loss(pos_scores, neg_scores)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch} BPR loss: {total_loss / STEPS_PER_EPOCH:.4f}")


# -----------------------------
# 7. Cache final embeddings once
# -----------------------------

@torch.no_grad()
def cache_lightgcn_embeddings(model):
    model.eval()
    users_emb, items_emb = model.propagate()
    return users_emb.detach(), items_emb.detach()


cached_user_emb, cached_item_emb = cache_lightgcn_embeddings(model)


# -----------------------------
# 8. LightGCN score functions
# -----------------------------

def lightgcn_score_fn(user_idx, game_idx_array):
    game_idx_array = np.asarray(game_idx_array, dtype=np.int64)

    with torch.no_grad():
        u = cached_user_emb[int(user_idx)]
        items = cached_item_emb[
            torch.tensor(game_idx_array, dtype=torch.long, device=DEVICE)
        ]

        scores = (items * u).sum(dim=1)

    return scores.cpu().numpy()


def predict_lightgcn_rating(user_idx, game_idx):
    raw_score = lightgcn_score_fn(user_idx, np.array([game_idx]))[0]

    scaled = rating_min + (rating_max - rating_min) / (1 + np.exp(-raw_score))

    return float(scaled)


# -----------------------------
# 9. RMSE evaluation
# -----------------------------

def rmse(preds, actuals):
    preds = np.asarray(preds)
    actuals = np.asarray(actuals)
    return float(np.sqrt(np.mean((preds - actuals) ** 2)))


def evaluate_rmse(eval_df, name="Validation"):
    preds = [
        predict_lightgcn_rating(int(row.user_idx), int(row.game_idx))
        for row in tqdm(eval_df.itertuples(), total=len(eval_df), desc=f"{name} RMSE")
    ]

    actuals = eval_df["rating"].values

    return rmse(preds, actuals)


val_rmse = evaluate_rmse(val, "Validation")
test_rmse = evaluate_rmse(test, "Test")

print("Validation RMSE:", val_rmse)
print("Test RMSE:", test_rmse)


# -----------------------------
# 10. Corrected ranking evaluation harness
# -----------------------------

def _sample_eval_subset(test_df, n=N_EVAL_USERS, seed=SEED):
    if n is None or n >= len(test_df):
        return test_df.reset_index(drop=True)

    return test_df.sample(n=n, random_state=seed).reset_index(drop=True)


def _build_user_seen_arrays(train_df):
    user_seen = {}

    for u, grp in train_df.groupby("user_idx", sort=False)["game_idx"]:
        user_seen[int(u)] = np.unique(grp.values.astype(np.int64))

    return user_seen


def evaluate_ranking(
    model_score_fn,
    test_df,
    train_df,
    n_negatives=99,
    k=10,
    seed=SEED,
    debug=False
):
    rng = np.random.RandomState(seed)

    eval_df = _sample_eval_subset(test_df, n=N_EVAL_USERS, seed=seed)
    user_seen = _build_user_seen_arrays(train_df)

    recalls = []
    ndcgs = []
    ranks = []

    users = eval_df["user_idx"].values.astype(np.int64)
    positives = eval_df["game_idx"].values.astype(np.int64)

    all_items = np.arange(N_GAMES, dtype=np.int64)

    for idx, (u, pos) in enumerate(
        tqdm(zip(users, positives), total=len(users), desc="Ranking eval")
    ):
        seen = user_seen.get(int(u), np.array([], dtype=np.int64))

        # Exclude training items and the positive eval item from negatives
        invalid = np.concatenate([seen, np.array([pos], dtype=np.int64)])
        valid_neg_pool = np.setdiff1d(all_items, invalid, assume_unique=False)

        if len(valid_neg_pool) < n_negatives:
            continue

        negs = rng.choice(valid_neg_pool, size=n_negatives, replace=False)

        candidates = np.concatenate([[pos], negs])
        labels = np.concatenate([[1], np.zeros(len(negs), dtype=np.int64)])

        # IMPORTANT FIX:
        # Shuffle candidates so the positive is not always at index 0.
        perm = rng.permutation(len(candidates))
        candidates = candidates[perm]
        labels = labels[perm]

        scores = model_score_fn(int(u), candidates)

        # Sort by score descending.
        # Add tiny random noise only for tie-breaking, not enough to change meaningful rankings.
        tie_breaker = rng.normal(0, 1e-12, size=len(scores))
        order = np.argsort(-(scores + tie_breaker))

        # Find rank position of the positive item after sorting.
        positive_rank = int(np.where(labels[order] == 1)[0][0])
        ranks.append(positive_rank)

        if positive_rank < k:
            recalls.append(1.0)
            ndcgs.append(1.0 / np.log2(positive_rank + 2))
        else:
            recalls.append(0.0)
            ndcgs.append(0.0)

        if debug and idx < 5:
            print("\n--- Debug example ---")
            print("User:", u)
            print("Positive item:", pos)
            print("Positive rank:", positive_rank)
            print("Positive score:", scores[labels == 1][0])
            print("Score min:", scores.min())
            print("Score max:", scores.max())
            print("Unique rounded scores:", len(np.unique(np.round(scores, 6))))
            print("Top-10 candidates:", candidates[order[:10]])
            print("Top-10 labels:", labels[order[:10]])

    return {
        "recall@10": float(np.mean(recalls)) if recalls else 0.0,
        "ndcg@10": float(np.mean(ndcgs)) if ndcgs else 0.0,
        "mean_rank": float(np.mean(ranks)) if ranks else None,
        "median_rank": float(np.median(ranks)) if ranks else None,
        "n_eval": len(recalls)
    }


# -----------------------------
# 11. Run corrected ranking evaluation
# -----------------------------

val_ranking = evaluate_ranking(
    model_score_fn=lightgcn_score_fn,
    test_df=val,
    train_df=train,
    n_negatives=99,
    k=10,
    seed=SEED,
    debug=True
)

test_ranking = evaluate_ranking(
    model_score_fn=lightgcn_score_fn,
    test_df=test,
    train_df=train,
    n_negatives=99,
    k=10,
    seed=SEED,
    debug=True
)

print("Validation ranking:", val_ranking)
print("Test ranking:", test_ranking)


# -----------------------------
# 12. Final results table
# -----------------------------

results = pd.DataFrame([
    {
        "Model": "LightGCN",
        "Val RMSE": val_rmse,
        "Test RMSE": test_rmse,
        "Val Recall@10": val_ranking["recall@10"],
        "Val NDCG@10": val_ranking["ndcg@10"],
        "Test Recall@10": test_ranking["recall@10"],
        "Test NDCG@10": test_ranking["ndcg@10"],
        "Val Mean Rank": val_ranking["mean_rank"],
        "Test Mean Rank": test_ranking["mean_rank"],
        "N Eval Users": test_ranking["n_eval"]
    }
])

results

2.5.1+cu121
True
12.1
1
Using device: cuda
      user  rating  game_id                  name  user_idx  game_idx
0  Hessu68     4.5    57422         Climate-Poker     52597      9435
1     Doel     9.0   120547           War & Peace     32710     11338
2    butze     6.0      915  Mystery of the Abbey    156861       736
3   dixijo     7.0      826             Cartagena    170065       676
4    pence     4.0   119407            Dixit Jinx    231508     11298
Index(['user', 'rating', 'game_id', 'name', 'user_idx', 'game_idx'], dtype='object')
Train: (18171881, 6)
Val: (272319, 6)
Test: (272319, 6)
Users: 272319
Games: 21802


Epoch 1/10: 100%|██████████| 2218/2218 [37:00<00:00,  1.00s/it]


Epoch 1 BPR loss: 0.6931


Epoch 2/10: 100%|██████████| 2218/2218 [37:07<00:00,  1.00s/it]


Epoch 2 BPR loss: 0.6931


...

Ranking eval:   3%|▎         | 32/1000 [00:00<00:03, 292.02it/s]


--- Debug example ---
User: 242545
Positive item: 1531
Positive rank: 20
Positive score: 0.0
Score min: 0.0
Score max: 0.0
Unique rounded scores: 1
Top-10 candidates: [11239  3686  6858 10904  4776  4652 17612 10898   411  1105]
Top-10 labels: [0 0 0 0 0 0 0 0 0 0]

--- Debug example ---
User: 44826
Positive item: 101
Positive rank: 74
Positive score: 0.0
Score min: 0.0
Score max: 0.0
Unique rounded scores: 1
Top-10 candidates: [14383 10718 12274 13811  5647 15732 11590  8114 11871  4887]
Top-10 labels: [0 0 0 0 0 0 0 0 0 0]

--- Debug example ---
User: 112880
Positive item: 1547
Positive rank: 18
Positive score: 0.0
Score min: 0.0
Score max: 0.0
Unique rounded scores: 1
Top-10 candidates: [ 4307  5115  4261  1577 17817  6798 18286  6477  4257 12262]
Top-10 labels: [0 0 0 0 0 0 0 0 0 0]

--- Debug example ---
User: 114018
Positive item: 337
Positive rank: 50
Positive score: 0.0
Score min: 0.0
Score max: 0.0
Unique rounded scores: 1
Top-10 candidates: [20224   333 11904 19314  8762 139

Ranking eval: 100%|██████████| 1000/1000 [00:01<00:00, 611.82it/s]


Validation ranking: {'recall@10': 0.082, 'ndcg@10': 0.037444197275421476, 'mean_rank': 49.327, 'median_rank': 47.0, 'n_eval': 1000}
Test ranking: {'recall@10': 0.108, 'ndcg@10': 0.047991078440632995, 'mean_rank': 49.312, 'median_rank': 49.0, 'n_eval': 1000}


,Model,Val RMSE,Test RMSE,Val Recall@10,Val NDCG@10,Test Recall@10,Test NDCG@10,Val Mean Rank,Test Mean Rank,N Eval Users
0,LightGCN,2.562214,2.561063,0.082,0.037444,0.108,0.047991,49.327,49.312,1000
